# state_conditioned_multi_objective_ytd_gate_v1

Preflight card for the next trainable candidate from `retention_ytd_training_objective_handoff_v1`. The candidate is not promoted and may not retain an actor until official `wf_2020`, `wf_2022`, `wf_2023_2024`, and YTD gates pass.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.config import load_profile
from src.soxl_quant.data import feature_columns_for_packs, feature_schema_signature
from src.soxl_quant.registry import load_profile_registry, validate_profile_registry

EXPERIMENT = "state_conditioned_multi_objective_ytd_gate_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
HANDOFF = ROOT / "runtime" / "model_selection" / "card" / "retention_ytd_training_objective_handoff_v1" / "selection.json"
RETENTION_GAP = ROOT / "runtime" / "model_selection" / "card" / "retention_gap_aggregate_v1" / "selection.json"

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


In [2]:
handoff = read_json(HANDOFF)
retention_gap = read_json(RETENTION_GAP)
profile = load_profile(EXPERIMENT)
registry_entry = load_profile_registry()[EXPERIMENT]
registry_issues = validate_profile_registry()
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)

preflight = {
    "experiment": EXPERIMENT,
    "profile_name": profile.name,
    "registry_status": registry_entry.status,
    "registry_baseline": registry_entry.baseline,
    "registry_keep_actor": registry_entry.keep_actor,
    "registry_keep_summary": registry_entry.keep_summary,
    "registry_notes_ref": registry_entry.notes_ref,
    "registry_issues": registry_issues,
    "feature_packs": list(profile.feature.packs),
    "feature_count": len(feature_columns),
    "feature_schema_signature": feature_schema_signature(profile.feature.packs, profile.name),
    "changed_layer": ["feature", "reward", "selection"],
    "canonical_baseline": handoff["canonical_baseline"],
    "direct_comparison_baseline": registry_entry.baseline,
    "reference_baseline": handoff["reference_baseline"],
    "target_window": handoff["target_thresholds"]["required_wf_windows"],
    "hard_wf_max_drawdown_pct": profile.promotion_gate.max_test_2022_drawdown,
    "min_wf_relative_outperformance_pp": profile.promotion_gate.min_wf_relative_outperformance,
    "ytd_current_best_alignment_pct": handoff["target_thresholds"]["required_current_best_alignment_ytd_pct"],
    "soxlbacktest_reference_ytd_return_pct": handoff["target_thresholds"]["soxlbacktest_reference_ytd_return_pct"],
    "safe_backtest_end": handoff["target_thresholds"]["safe_backtest_end"],
    "closest_success_case": retention_gap["best_wf_drawdown_pass_candidate_by_ytd"]["candidate"],
    "closest_success_gap": retention_gap["best_wf_drawdown_pass_candidate_by_ytd"]["failures"],
    "closest_failure_case": retention_gap["best_complete_candidate"]["candidate"],
    "closest_failure_gap": retention_gap["best_complete_candidate"]["failures"],
    "hypothesis": "State-conditioned reward plus trend-preservation and drawdown-drift terms can keep the best complete candidate's participation while reducing WF drawdown below 20pct.",
    "acceptance_criteria": [
        "official wf_2020/wf_2022/wf_2023_2024 metrics are present",
        "worst official WF MaxDD < 20pct",
        "YTD return >= SOXLBacktest buy/sell reference and >= 95pct of current best",
        "checkpoint selector table includes retention_ytd_loss and wf_drawdown_loss",
        "drawdown interval audit exists for selected checkpoint",
    ],
    "expected_tradeoff": "More conservative stress-state positioning may reduce peak YTD return; trend-preservation terms are intended to prevent the low-drawdown underparticipation failure mode.",
    "failure_mode": "Either WF drawdown stays above 20pct like mode_conditioned_rebound_reward_train_v2, or the new drawdown terms collapse YTD participation like wf_pass_ytd18_15_15_02.",
}

preflight


{'experiment': 'state_conditioned_multi_objective_ytd_gate_v1',
 'profile_name': 'state_conditioned_multi_objective_ytd_gate_v1',
 'registry_status': 'candidate',
 'registry_baseline': 'mode_conditioned_rebound_reward_train_v2',
 'registry_keep_actor': False,
 'registry_keep_summary': True,
 'registry_notes_ref': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/state_conditioned_multi_objective_ytd_gate_v1.ipynb',
 'registry_issues': [],
 'feature_packs': ['vix_term',
  'transition_signals',
  'soxx_spread_oscillation'],
 'feature_count': 79,
 'feature_schema_signature': '2f64801809a1',
 'changed_layer': ['feature', 'reward', 'selection'],
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'mode_conditioned_rebound_reward_train_v2',
 'reference_baseline': 'soxlbacktest_policy_reimplementation_v1',
 'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'hard_wf_max_drawdown_pct': 20.0,
 'min_wf_relative_outperfor

In [3]:
assert preflight["registry_status"] == "candidate"
assert preflight["registry_keep_actor"] is False
assert preflight["direct_comparison_baseline"] == "mode_conditioned_rebound_reward_train_v2"
assert preflight["hard_wf_max_drawdown_pct"] <= 20.0
assert preflight["min_wf_relative_outperformance_pp"] >= 30.0
assert profile.reward.state_conditioned_reward_enabled is True
assert profile.reward.drawdown_drift_reward_enabled is True
assert profile.reward.trend_preservation_floor_enabled is True
assert profile.selection_score.use_pareto is True
assert profile.checkpoint_retention.enabled is True
assert registry_issues == []

created_at = datetime.now(timezone.utc).isoformat()
summary = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": "candidate_profile_ready_for_smoke_training_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_next": True,
    "profile_yaml": f"runtime/model_selection/card/{EXPERIMENT}/profile.yaml",
    "profile_preflight": f"runtime/model_selection/card/{EXPERIMENT}/profile_preflight.json",
    "source_handoff": str(HANDOFF.relative_to(ROOT)),
    "source_retention_gap": str(RETENTION_GAP.relative_to(ROOT)),
    "target_thresholds": {
        "required_wf_windows": preflight["target_window"],
        "wf_max_drawdown_hard_gate_pct": preflight["hard_wf_max_drawdown_pct"],
        "official_internal_relative_return_floor_pp": preflight["min_wf_relative_outperformance_pp"],
        "soxlbacktest_reference_ytd_return_pct": preflight["soxlbacktest_reference_ytd_return_pct"],
        "required_current_best_alignment_ytd_pct": preflight["ytd_current_best_alignment_pct"],
        "safe_backtest_end": preflight["safe_backtest_end"],
    },
    "candidate_profile": {
        "name": profile.name,
        "baseline": registry_entry.baseline,
        "feature_packs": preflight["feature_packs"],
        "feature_count": preflight["feature_count"],
        "feature_schema_signature": preflight["feature_schema_signature"],
        "changed_layer": preflight["changed_layer"],
    },
    "next_required_artifacts": handoff["required_next_artifacts"],
}
selection = dict(summary)
selection.update({
    "closest_success_case": preflight["closest_success_case"],
    "closest_failure_case": preflight["closest_failure_case"],
    "hypothesis": preflight["hypothesis"],
    "acceptance_criteria": preflight["acceptance_criteria"],
    "expected_tradeoff": preflight["expected_tradeoff"],
    "failure_mode": preflight["failure_mode"],
})

(CARD / "profile_preflight.json").write_text(json.dumps(preflight, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary


{'experiment': 'state_conditioned_multi_objective_ytd_gate_v1',
 'created_at': '2026-06-13T14:50:21.070060+00:00',
 'decision': 'candidate_profile_ready_for_smoke_training_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_next': True,
 'profile_yaml': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/profile.yaml',
 'profile_preflight': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/profile_preflight.json',
 'source_handoff': 'runtime/model_selection/card/retention_ytd_training_objective_handoff_v1/selection.json',
 'source_retention_gap': 'runtime/model_selection/card/retention_gap_aggregate_v1/selection.json',
 'target_thresholds': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'wf_max_drawdown_hard_gate_pct': 20.0,
  'official_internal_relative_return_floor_pp': 30.0,
  'soxlbacktest_reference_ytd_return_pct': 132.077978,
  'required_current_best_alignment_

## Conclusion

Decision: `candidate_profile_ready_for_smoke_training_no_promotion`.

This card creates the next trainable profile requested by the retention handoff. It is deliberately registered with `keep_actor: false` so dashboard/model replay discovery cannot surface it as usable before official evidence exists. The candidate differs from the closest failed reward diagnostics by adding an explicit feature schema (`vix_term`, `transition_signals`, `soxx_spread_oscillation`), Pareto checkpoint selection, a 20% max-drawdown gate, drawdown-drift penalty, and trend-preservation floor.

Next action: run smoke training first, then official `wf_2020`, `wf_2022`, `wf_2023_2024`, dashboard/YTD replay, checkpoint selector table with `retention_ytd_loss` and `wf_drawdown_loss`, and selected-checkpoint drawdown interval audit. Promotion and actor retention remain blocked until those artifacts pass.